<a href="https://colab.research.google.com/github/mariolopezguasp/SP500Prediction/blob/main/3ModeloNeuronal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv('dataset_ia_log_returns_10y.csv', index_col=0, parse_dates=True)


X = df.iloc[:-1].values
y = df.iloc[1:].values

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

# Initialize and apply MinMaxScaler to features (X)
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_val = scaler_X.transform(X_val)
X_test = scaler_X.transform(X_test)

print(f"Train size: {X_train.shape}, Val size: {X_val.shape}, Test size: {X_test.shape}")

Train size: (1757, 12), Val size: (376, 12), Test size: (377, 12)


In [3]:
import os

if not os.path.exists('modelos'):
    os.makedirs('modelos')
    print("Created directory: modelos")
else:
    print("Directory 'modelos' already exists.")


Created directory: modelos


In [4]:
%%writefile modelos/modelo_simple.py

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def get_simple_nn(input_dim, output_dim):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_dim,)),
        Dense(32, activation='relu'),
        Dense(output_dim, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


Writing modelos/modelo_simple.py


In [5]:
import sys
import os
import json
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Añadir el path para importar el modelo
sys.path.append(os.path.abspath('.'))
from modelos.modelo_simple import get_simple_nn

# Create the model
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
model = get_simple_nn(input_dim, output_dim)

num_params = model.count_params()
print(f"Número de Parámetros del Modelo de Nivel 1: {num_params}")

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), verbose=0)

# Make predictions
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Calculate MSE
mse_train = mean_squared_error(y_train, y_train_pred)
mse_val = mean_squared_error(y_val, y_val_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {mse_train:.6f}")
print(f"Validation MSE: {mse_val:.6f}")
print(f"Test MSE: {mse_test:.6f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Número de Parámetros del Modelo de Nivel 1: 3308
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Train MSE: 0.000357
Validation MSE: 0.000308
Test MSE: 0.000399


In [6]:
rmse_train_nn = np.sqrt(mse_train)
rmse_val_nn = np.sqrt(mse_val)
rmse_test = np.sqrt(mse_test)

print("--- Root Mean Squared Error (RMSE) - Red Neuronal Simple ---")
print(f"Train RMSE: {rmse_train_nn:.6f} (Desviación media del modelo: {rmse_train_nn*100:.2f}%)")
print(f"Validation RMSE: {rmse_val_nn:.6f} (Desviación media del modelo: {rmse_val_nn*100:.2f}%)")
print(f"Test RMSE: {rmse_test:.6f} (Desviación media del modelo: {rmse_test*100:.2f}%)")
print("-" * 60)

--- Root Mean Squared Error (RMSE) - Red Neuronal Simple ---
Train RMSE: 0.018896 (Desviación media del modelo: 1.89%)
Validation RMSE: 0.017558 (Desviación media del modelo: 1.76%)
Test RMSE: 0.019979 (Desviación media del modelo: 2.00%)
------------------------------------------------------------


In [7]:
import numpy as np

# --- Directional Accuracy for Training Set ---
signo_real_train = np.sign(y_train)
signo_pred_train = np.sign(y_train_pred)
accuracy_direccional_train = np.mean(signo_real_train == signo_pred_train)

# --- Directional Accuracy for Validation Set ---
signo_real_val = np.sign(y_val)
signo_pred_val = np.sign(y_val_pred)
accuracy_direccional_val = np.mean(signo_real_val == signo_pred_val)

# --- Directional Accuracy for Test Set ---
signo_real_test = np.sign(y_test)
signo_pred_test = np.sign(y_test_pred)
accuracy_direccional_test = np.mean(signo_real_test == signo_pred_test)

print("--- Accuracy Direccional ---")
print(f"Train Directional Accuracy: {accuracy_direccional_train*100:.2f}%")
print(f"Validation Directional Accuracy: {accuracy_direccional_val*100:.2f}%")
print(f"Test Directional Accuracy: {accuracy_direccional_test*100:.2f}%")

--- Accuracy Direccional ---
Train Directional Accuracy: 52.92%
Validation Directional Accuracy: 54.14%
Test Directional Accuracy: 52.14%
